# Backtesting temporal — AUC por ola
Analiza los resultados reales del campeón fijo, sin reentrenar. Ejecutar main con backtesting=true primero. En Colab subir backtesting_results.json; localmente indicar su ruta. Una ola sin ambas clases tiene AUC no definido y no entra en el promedio.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RESULTS_PATH = ""  # Ruta local opcional al JSON
if not RESULTS_PATH:
    from google.colab import files
    uploaded = files.upload()
    RESULTS_PATH = next(name for name in uploaded if name.endswith(".json"))
try:
    results = json.loads(Path(RESULTS_PATH).read_text(encoding="utf-8-sig"))
except UnicodeDecodeError:
    results = json.loads(Path(RESULTS_PATH).read_text(encoding="cp1252"))
assert len(results["waves"]) >= 4
assert results["model_refitted"] is False
print("Campeón:", results["champion"])
print("Corte:", results["cutoff_period"] or results["cutoff_file"])
print("Cronología:", results["chronology_note"])
print("AUC de referencia:", results["reference_auc"])

In [ ]:
table = pd.DataFrame(results["waves"])
display(table[["wave", "file", "period", "rows_evaluated", "auc", "decay_percent", "status", "prediction_seconds"]])
print("AUC promedio (olas válidas):", results["mean_auc_valid_waves"])
print("Olas con AUC definido:", results["valid_waves"])

In [ ]:
def graficar_auc(results, output_path=None):
    import matplotlib.pyplot as plt
    rows = results['waves']
    x = [row['wave'] for row in rows]
    auc = [row['auc'] if row['auc'] is not None else np.nan for row in rows]
    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.plot(x, auc, marker='o', linewidth=2, label='AUC mensual del campeón')
    ax.axhline(results['reference_auc'], color='tab:orange', linestyle='--', label='AUC de validación')
    ax.axhline(.5, color='gray', linestyle=':', label='Referencia AUC = 0.5')
    ax.set_xticks(x, [f'T+{r["wave"]}\n{r["period"] or r["file"]}' for r in rows])
    ax.set_ylim(0, 1.03); ax.set_xlabel('Ola / mes futuro'); ax.set_ylabel('AUC ROC')
    ax.set_title(f'Backtesting temporal — {results["champion"]}')
    for r in rows:
        if r['auc'] is not None: ax.annotate(f'{r["auc"]:.4f}', (r['wave'], r['auc']), xytext=(0, 8), textcoords='offset points', ha='center')
    ax.grid(alpha=.25); ax.legend(loc='best'); fig.tight_layout()
    if output_path: fig.savefig(output_path, dpi=160, bbox_inches='tight')
    return fig

fig = graficar_auc(results, "auc_por_ola.png")
plt.show()

## Interpretación
La línea naranja representa el AUC de validación del campeón. Descensos sostenidos pueden indicar pérdida de discriminación temporal; esta gráfica por sí sola no identifica su causa. Decay positivo indica descenso y negativo mejora. Diferencias también dependen del muestreo y la composición mensual. No hay intervalos de confianza. Las fechas no verificadas se declaran en el reporte.